#### IMPORT PYSAPRK PACKAGES

In [0]:
from pyspark.sql.functions import to_date, to_timestamp, col, when, date_format, current_timestamp
from pyspark.sql import functions as F

In [0]:
refunds_df = spark.table('HIVE_METASTORE.BRONZE.REFUNDS')
casting_refunds_df = (refunds_df.select(
                                        'refund_id',
                                        'payment_id',
                                        'refund_reason',
                                        to_date(col("refund_timestamp"), "yyyy-MM-dd").alias('refund_date'),
                                        date_format(col("refund_timestamp"), "HH:mm:ss").alias('refund_time'),
                                        'refund_amount')
                                        )

display(casting_refunds_df)

#### EXTRACT `REFUND_CATEGORY` & `REFUNDED_BY`

In [0]:
refund_reason_refunds_df =  casting_refunds_df.withColumn("refund_category", F.split(F.col("refund_reason"), ":")[0]) \
                                      .withColumn("refunded_by", F.split(F.col("refund_reason"), ":")[1]) \
                                      .drop('refund_reason')

display(refund_reason_refunds_df)

In [0]:
refund_reason_refunds_final_df = refund_reason_refunds_df.selectExpr(
                                                    'refund_id', 'payment_id', 'refund_date', 
                                                    'refund_time', 'refund_amount', 'refund_category as refund_reason', 
                                                    'refunded_by')
display(refund_reason_refunds_final_df)

#### WRITE TRANSFORMED DATA TO SILVER SCHEMA
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: SILVER
3. TABLE NAME: REFUNDS_DELTA


In [0]:
refund_reason_refunds_final_df.writeTo('gizmo.silver.refunds_delta').createOrReplace()

#### VALIDATE AND QUERY `GIZMO.SILVER.REFUNDS_DELTA`

In [0]:
%sql
SELECT * FROM GIZMO.SILVER.REFUNDS_DELTA

In [0]:
%python
dbutils.notebook.exit("REFUNDS LOADED INTO GIZMO.SILVER.REFUNDS_DELTA")